# Tutorial 5: Train your own LLMs
### **Course Name:** CSC6052/5051/4100/DDA6307/MDS5110 Natural Language Processing




This notebook guide provides a comprehensive overview of using the `transformers` Python package to efficiently train a custom model. It covers the following techniques:

1. Load Model, Tokenizer and Template for Chat Model.
2. Process Data for Training.
2. Train Model with Qlora.
4. Evaluate Model's performance.
5. Save and Deploy Trained Model.

## Preliminary Preparation

Before proceeding with model training, ensure your environment is properly configured by following these steps:

1. Install the necessary Python packages.
2. Import the required libraries.

%pip install -q h5py typing-extensions wheel \
%pip install -q -U bitsandbytes \
%pip install -q -U git+https://github.com/huggingface/transformers.git \
%pip install -q -U git+https://github.com/huggingface/peft.git \
%pip install -q -U git+https://github.com/huggingface/accelerate.git \
%pip install -q datasets \

In [1]:
!nvidia-smi

Sun Mar 30 20:42:01 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.183.01             Driver Version: 535.183.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     Off | 00000000:17:00.0 Off |                  N/A |
|  0%   27C    P8              18W / 250W |      6MiB / 22528MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--


## Load Pre-trained model and tokenizer

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoConfig

model_id = "Qwen/Qwen2.5-7B-Instruct"

# 加载模型配置
config = AutoConfig.from_pretrained(model_id)

# 配置量化
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,  # 激活嵌套量化
    bnb_4bit_quant_type="nf4",  # 量化类型
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 加载模型
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0}
)

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(model_id)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## Preprocess the quantized model for training

In [3]:
from peft import prepare_model_for_kbit_training

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [4]:
from peft import LoraConfig, get_peft_model

# You can try differnt parameter-effient strategy for model trianing, for more info, please check https://github.com/huggingface/peft
config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, config)

## Chat Template Usage

In [5]:
from jinja2 import Template
template = Template(tokenizer.chat_template)
message = "Please introduce yourself"
print(f"message:\n{message}\n")
message_send_to_model=template.render(messages=[{"role": "user", "content": message}],bos_token=tokenizer.bos_token,add_generation_prompt=True)
print(f"message_send_to_model:\n{message_send_to_model}")

message:
Please introduce yourself

message_send_to_model:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Please introduce yourself<|im_end|>
<|im_start|>assistant



In [6]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# 定义模板
template = Template(tokenizer.chat_template)

@torch.no_grad()
def generate(prompt):
    # 渲染模板
    modelInput = template.render(
        messages=[{"role": "user", "content": prompt}],
        bos_token=tokenizer.bos_token,
        add_generation_prompt=True
    )
    print("-" * 80)
    print(f"model_input_string:\n{modelInput}")

    # 编码输入
    input_ids = tokenizer.encode(modelInput, add_special_tokens=False, return_tensors='pt').to(device)

    # 生成输出
    outputs = model.generate(input_ids, max_length=512, do_sample=True, temperature=0.7)

    # 解码输出
    model_return_string = tokenizer.decode(outputs[0], skip_special_tokens=False)
    print("-" * 80)
    print(f"model_return_string:\n{model_return_string}")

    # 提取生成的文本
    generated_ids = outputs[:, input_ids.shape[1]:]
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return generated_text

# 测试生成
query = "Please introduce yourself"
print("-" * 80)
print(f"query:\n{query}")
response = generate(query)
print("-" * 80)
print(f"response:\n{response}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--------------------------------------------------------------------------------
query:
Please introduce yourself
--------------------------------------------------------------------------------
model_input_string:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Please introduce yourself<|im_end|>
<|im_start|>assistant

--------------------------------------------------------------------------------
model_return_string:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Please introduce yourself<|im_end|>
<|im_start|>assistant
Hello! I'm Qwen, an AI assistant created by Alibaba Cloud. I'm here to help with a wide range of tasks such as answering questions, providing information on various topics, writing assistance, and more. I aim to be helpful, informative, and courteous in all my interactions. How can I assist you today?<|im_end|>
-----------------------

## Data Preparation

Let's load a common dataset, english quotes, to fine tune our model on famous quotes.

In [7]:
from datasets import load_dataset

# 加载数据集
dataset = load_dataset("FreedomIntelligence/Huatuo26M-Lite")

# 映射数据集，将每个样本转换为对话格式
dataset = dataset['train'].map(
    lambda sample: {
        "conversations": [
            {"from": "human", "value": sample['question']},
            {"from": "gpt", "value": sample['answer']}
        ]
    },
    batched=False
)

Using the latest cached version of the dataset since FreedomIntelligence/Huatuo26M-Lite couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/zcai/.cache/huggingface/datasets/FreedomIntelligence___huatuo26_m-lite/default/0.0.0/90ce61699e90568db82cdce4c4035c6918d70aa6 (last modified on Sun Mar 30 01:41:29 2025).


In [8]:
from torch.utils.data import random_split
train_dataset_size, val_dataset_size = 40, 8
train_dataset, val_dataset, _ = random_split(dataset, [train_dataset_size, val_dataset_size, len(dataset)-train_dataset_size-val_dataset_size])
print(train_dataset[0]['conversations'])

[{'from': 'human', 'value': '我家宝宝今天出生15天，昨天晚上开始吐奶，是喷出来那种，今天下午吃过奶后拍嗝把吃的都吐了，间隔一小时后再次喂奶又全部喷了出来，请问这是什么原因？'}, {'from': 'gpt', 'value': '宝宝吐奶是很常见的，但是如果频繁吐奶，需要注意是否有其他问题。可能是宝宝吃奶过快或者吃奶姿势不正确，也可能是宝宝消化系统还没有完全发育好，需要一些时间来适应。建议妈妈在喂奶时注意宝宝的吃奶姿势，让宝宝吃得慢一些，同时也要注意宝宝的消化情况，如果宝宝出现腹泻、发热等症状，建议及时就医。'}]


### Customized Dataset
Create a specialized dataset class named "InstructionDataset" designed to handle our custom dataset.

In [9]:
import torch
import transformers
from typing import Dict, Sequence, List
from torch.utils.data import Dataset
from dataclasses import dataclass

IGNORE_INDEX = -100  # 定义忽略索引

def preprocess(sources: List[Dict], tokenizer: transformers.PreTrainedTokenizer) -> Dict:
    """
    预处理函数，将输入的文本数据转换为模型可用的 input_ids 和 labels。

    Args:
        sources (List[Dict]): 包含对话数据的列表，每个元素是一个字典，包含 'conversations' 键。
        tokenizer (transformers.PreTrainedTokenizer): 用于分词的 tokenizer。

    Returns:
        Dict: 包含 input_ids 和 labels 的字典。
    """
    messages = []
    max_seq_len = 512  # 最大序列长度

    for source in sources:
        input_ids = []
        labels = []

        # 检查 source 是否包含 'conversations' 键
        if "conversations" not in source or not source["conversations"]:
            print(f"Invalid source, missing 'conversations': {source}")
            continue

        # 遍历对话内容
        for message in source["conversations"]:
            if message["from"] == "human":
                # 人类输入的文本
                input_ids += tokenizer.encode(message["value"], add_special_tokens=False)
                labels += [IGNORE_INDEX] * len(input_ids)
            elif message["from"] == "gpt":
                # 模型生成的文本
                response_ids = tokenizer.encode(message["value"], add_special_tokens=False)
                input_ids += response_ids
                labels += response_ids

        # 如果 input_ids 为空，跳过该样本
        if len(input_ids) == 0:
            print("Empty input_ids, skipping sample.")
            continue

        # 截断到最大长度
        input_ids = input_ids[-max_seq_len:]
        labels = labels[-max_seq_len:]

        messages.append({"input_ids": input_ids, "labels": labels})

    if len(messages) == 0:
        print("No valid messages found.")
        return {"input_ids": torch.LongTensor([]), "labels": torch.LongTensor([])}

    # 生成最终张量
    input_ids = [torch.LongTensor(item["input_ids"]) for item in messages]
    labels = [torch.LongTensor(item["labels"]) for item in messages]

    return {
        "input_ids": input_ids,
        "labels": labels,
    }


class InstructDataset(Dataset):
    """
    自定义数据集类，用于加载和处理对话数据。
    """
    def __init__(self, data: Sequence[Dict], tokenizer: transformers.PreTrainedTokenizer) -> None:
        """
        初始化数据集。

        Args:
            data (Sequence[Dict]): 数据列表，每个元素是一个字典，包含 'conversations' 键。
            tokenizer (transformers.PreTrainedTokenizer): 用于分词的 tokenizer。
        """
        super().__init__()
        self.tokenizer = tokenizer
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index) -> Dict[str, torch.Tensor]:
        """
        获取指定索引的数据样本。

        Args:
            index (int): 数据索引。

        Returns:
            Dict[str, torch.Tensor]: 包含 input_ids 和 labels 的字典。
        """
        sources = self.data[index]
        if isinstance(index, int):
            sources = [sources]
        try:
            data_dict = preprocess(sources, self.tokenizer)
        except KeyError as e:
            print(f"Missing key in source: {e}")
            raise
        if isinstance(index, int):
            data_dict = dict(input_ids=data_dict["input_ids"][0], labels=data_dict["labels"][0])
        return data_dict


@dataclass
class DataCollatorForSupervisedDataset:
    """
    数据整理器，用于将多个样本整理成批次。
    """
    tokenizer: transformers.PreTrainedTokenizer
    device: torch.device  # 添加设备参数

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        """
        整理批次数据。

        Args:
            instances (Sequence[Dict]): 数据样本列表。

        Returns:
            Dict[str, torch.Tensor]: 包含批次 input_ids、labels 和 attention_mask 的字典。
        """
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id
        )
        labels = torch.nn.utils.rnn.pad_sequence(
            labels,
            batch_first=True,
            padding_value=IGNORE_INDEX
        )
        attention_mask = input_ids.ne(self.tokenizer.pad_token_id)

        # 将张量移动到指定设备
        return dict(
            input_ids=input_ids.to(self.device),
            labels=labels.to(self.device),
            attention_mask=attention_mask.to(self.device),
        )

In [10]:
train_dataset = InstructDataset(train_dataset, tokenizer)
val_dataset = InstructDataset(val_dataset, tokenizer)
data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer, device=device)

In [11]:
sample_data = train_dataset[9]
IGNORE_INDEX = -100

print("=" * 80)
print("Debugging: Full sample data")
print(sample_data)  # 打印完整的样本数据
print("=" * 80)

# 检查是否包含 input_ids 和 labels
if "input_ids" in sample_data and "labels" in sample_data:
    print(f"Input_ids:\n{sample_data['input_ids']}")
    print(f"Label_ids:\n{sample_data['labels']}")
else:
    print("Missing 'input_ids' or 'labels' in sample_data.")

# 检查 input_ids 和 labels 的长度
if "input_ids" in sample_data:
    print(f"Length of input_ids: {len(sample_data['input_ids'])}")
if "labels" in sample_data:
    print(f"Length of labels: {len(sample_data['labels'])}")

# 检查是否存在空的 input_ids 或 labels
if len(sample_data.get("input_ids", [])) == 0:
    print("Error: input_ids is empty.")
if len(sample_data.get("labels", [])) == 0:
    print("Error: labels is empty.")

Debugging: Full sample data
{'input_ids': tensor([ 35946,     18,     17,  92015,  34187,   3837, 107307, 113064, 102781,
        101561,   3837,  99172, 101051, 100232, 100638,   3837, 101056, 105432,
        103920, 106481, 104355,  59151,   3837, 110277,  36587,  73670, 104160,
          3837, 100131, 104047, 109513, 104160,   1773, 113064, 102781, 101561,
         20412,  86402,  99245, 101160, 101036, 113064, 102781, 101561,  20412,
        101968, 112170,  72448, 102716, 101160,   3837,  57218, 108122, 100741,
          5373, 114539,  99559,  33108, 107142,  49567, 101063,   1773, 113064,
        102781, 101561,  18830,  99584, 105178, 114210,  53930,  17177,   1773,
        101899,  75768, 108747, 108723, 102185,   5373, 102781, 101561,   9370,
        105155,   5373, 105130,   5373,  92032,   5373, 104569, 101149,   3837,
        101034,  64471, 104465, 103042,  98380,  33108, 108723, 107869, 110276,
          1773]), 'labels': tensor([  -100,   -100,   -100,   -100,   -100,   

In [12]:
print(f"Input:\n{tokenizer.decode(sample_data['input_ids'])}")
print("-" * 80)
N_id = tokenizer.encode("N", add_special_tokens= False)[0]
print(f"Label:\n{tokenizer.decode([N_id if x == -100 else x for x in sample_data['labels']])}")
print("=" * 80)

Input:
我32岁了，得了卵巢囊肿，想得到根本解决，之前怀孕的时候体检出来的结果，大夫说可以手术，但是也可以不做手术。卵巢囊肿是种什么疾病呢卵巢囊肿是女性生殖系统常见的疾病，与遗传因素、内分泌情况和生活方式等有关。卵巢囊肿有良性和恶性之分。治疗方式取决于患者的年龄、囊肿的性质、部位、大小、生长速度，以及是否保留生育功能和患者的意愿等因素。
--------------------------------------------------------------------------------
Label:
NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN卵巢囊肿是女性生殖系统常见的疾病，与遗传因素、内分泌情况和生活方式等有关。卵巢囊肿有良性和恶性之分。治疗方式取决于患者的年龄、囊肿的性质、部位、大小、生长速度，以及是否保留生育功能和患者的意愿等因素。


## Training

### General Training Hyperparameters

In [18]:
# Set training parameters
training_arguments = transformers.TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    optim='adamw_torch',  # 使用有效的优化器名称
    save_steps=10,
    logging_steps=10,
    learning_rate=5e-5,  # 调整学习率
    weight_decay=0.001,
    max_steps=1000,  # 限制最大训练步数
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    gradient_checkpointing=False,  # 如果显存充足，可以关闭
    report_to="none",
    no_cuda=False)

In [22]:
from transformers import Trainer
from torch.utils.data import DataLoader

class CustomTrainer(Trainer):
    def get_train_dataloader(self) -> DataLoader:
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            shuffle=True,
            collate_fn=self.data_collator,
            pin_memory=False  # 禁用 pin_memory
        )

    def get_eval_dataloader(self, eval_dataset=None) -> DataLoader:
        eval_dataset = eval_dataset if eval_dataset is not None else self.eval_dataset
        return DataLoader(
            eval_dataset,
            batch_size=self.args.eval_batch_size,
            shuffle=False,
            collate_fn=self.data_collator,
            pin_memory=False  # 禁用 pin_memory
        )

    def inspect_dataloader(self):
        # 检查训练数据加载器
        print("Inspecting train dataloader...")
        train_dataloader = self.get_train_dataloader()
        for batch in train_dataloader:
            print("Train batch:")
            print(batch)
            break  # 只检查一个批次

        # 检查验证数据加载器
        print("Inspecting eval dataloader...")
        eval_dataloader = self.get_eval_dataloader()
        for batch in eval_dataloader:
            print("Eval batch:")
            print(batch)
            break  # 只检查一个批次

In [25]:
def training_step(self, model, inputs):
    print("Input devices:")
    for key, value in inputs.items():
        if isinstance(value, torch.Tensor):
            print(f"{key}: {value.device}")
    return super().training_step(model, inputs)

In [36]:
# 确保设备一致性
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 将模型移动到设备
model = model.to(device)

# 确保模型处于训练模式
model.train()

# 初始化 Trainer
print("Initializing Trainer...")
trainer = CustomTrainer(
    model=model,
    args=training_arguments,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

# Start training
trainer.train()

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Using device: cuda
Initializing Trainer...


RuntimeError: chunk expects at least a 1-dimensional tensor

In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

model.print_trainable_parameters()

trainable params: 2,523,136 || all params: 7,618,139,648 || trainable%: 0.0331


Once the training is completed, we can evaluate our model and get its perplexity on the validation set like this:

In [ ]:
import math
%pip install -q -U git+https://github.com/huggingface/accelerate.git
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  error: subprocess-exited-with-error
  
  × git clone --filter=blob:none --quiet https://github.com/huggingface/accelerate.git /tmp/pip-req-build-nf04wtnq did not run successfully.
  │ exit code: 128
  ╰─> [1 lines of output]
      fatal: 无法访问 'https://github.com/huggingface/accelerate.git/'：Failed to connect to 0.0.0.0 port 7890 after 0 ms: 连接被拒绝
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× git clone --filter=blob:none --quiet https://github.com/huggingface/accelerate.git /tmp/pip-req-build-nf04wtnq did not run successfully.
│ exit code: 128
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
Note: you may need to restart the kernel to use updated packages.


RuntimeError: cannot pin 'torch.cuda.LongTensor' only dense CPU tensors can be pinned

## Save Trained LoRA

In [ ]:
!pwd
output_path = "ilora"
trainer.save_model(output_path)

/content


### Test the trained model

In [ ]:
template = Template(tokenizer.chat_template)
@torch.no_grad()
def generate(prompt):
    modelInput = template.render(messages=[{"role": "user", "content": prompt}],bos_token= tokenizer.bos_token,add_generation_prompt=True)
    input_ids = tokenizer.encode(modelInput, add_special_tokens=False, return_tensors='pt').to("cuda:0")
    outputs = model.generate(input_ids, temperature=1.0)
    model_return_string = tokenizer.decode(*outputs, skip_special_tokens=False)
    print("-"*80)
    print(f"model_return_string:\n{model_return_string}")
    generated_ids = outputs[:, input_ids.shape[1]:]
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
    return generated_text

query = "I get hit"
print(f"query:\n{query}")
response = generate(query)
print("-"*80)
print(f"response:\n{response}")

query:
I get hit
--------------------------------------------------------------------------------
model_return_string:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
I get hit<|im_end|>
<|im_start|>assistant
I'm sorry to hear that! Are you okay? If this is a physical situation, please ensure
--------------------------------------------------------------------------------
response:
I'm sorry to hear that! Are you okay? If this is a physical situation, please ensure


# Clean GPU Memory

In [ ]:
# Empty VRAM
# del model
# del trainer
import gc
import torch
torch.cuda.empty_cache()
gc.collect()
gc.collect()

0

In [ ]:
!nvidia-smi

Fri Mar 21 04:12:52 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             32W /   70W |     320MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Load the trained model back and integrate the trained LoRA within.

In [ ]:
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained(model_id, load_in_8bit=True, device_map={"":0})
model = PeftModel.from_pretrained(model, output_path)
model = model.merge_and_unload()
model.config.max_length = 512
model.eval()

tokenizer = transformers.AutoTokenizer.from_pretrained(model_id, padding_side="left")
# tokenizer.pad_token = tokenizer.unk_token


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/bnb.py:85: UserWarning: Merge lora module to 8-bit linear may get different generations due to rounding errors.
  warnings.warn(


## Answer generation

In [ ]:
@torch.no_grad()
def generate(prompts):
    model_inputs = [template.render(messages=[{"role": "user", "content": prompt}], bos_token=tokenizer.bos_token, add_generation_prompt=True) for prompt in prompts]
    input_ids = tokenizer(model_inputs, add_special_tokens=False, return_tensors='pt', padding=True).to("cuda:0")

    outputs = model.generate(input_ids.input_ids, attention_mask=input_ids.attention_mask, max_new_tokens=100)

    generated_texts = []
    for i in range(len(prompts)):
        generated_ids = outputs[i, input_ids.input_ids.shape[1]:]
        generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        generated_texts.append(generated_text)

    return generated_texts

# test
print("\n\n".join(generate(["I get hit", "Who are you?"])))


I'm sorry to hear that you're feeling hurt. Can you provide more context about what happened? Is this related to a physical injury or emotional distress? It's important to take care of yourself. If it's a physical injury, please seek medical attention if necessary. For emotional support, talking about your feelings can be very helpful.

I am Qwen, a large language model created by Alibaba Cloud. I'm here to assist with a wide variety of tasks and answer any questions you might have! How can I help you today?


## Evaluate a trained model on a given test dataset

In [ ]:
!wget https://NLP-course-cuhksz.github.io/Assignments/Assignment1/task1/data/1.exam.json

--2025-03-21 04:15:24--  https://nlp-course-cuhksz.github.io/Assignments/Assignment1/task1/data/1.exam.json
Resolving nlp-course-cuhksz.github.io (nlp-course-cuhksz.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to nlp-course-cuhksz.github.io (nlp-course-cuhksz.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 86227 (84K) [application/json]
Saving to: ‘1.exam.json’

1.exam.json         100%[===================>]  84.21K  --.-KB/s    in 0.02s   

2025-03-21 04:15:24 (3.60 MB/s) - ‘1.exam.json’ saved [86227/86227]



In [ ]:
import json

with open('1.exam.json') as f:
  data = json.load(f)
  data = data[:20] # just for demo

print(data[0])

{'question': '27. 根据国家药品监督管理局，公安部，国家卫⽣健康委员会的有关规定，⼜服固体制剂每剂量单位含羟考酮碱不超过5毫克，且不含其他⿇醉药品，精神药品或者药品类易制毒化学品的复⽅制剂列⼊（）。', 'option': {'A': '含⿇醉药品复⽅制剂的管理', 'B': '第⼆类精神药品管理', 'C': '第⼀类精神药品管理', 'D': '医疗⽤毒性药品管理', 'E': ''}, 'analysis': '⼜服固体制剂每剂量单位含羟考酮碱不超过5毫克，且不含其他⿇醉药品、精神药品或药品类易制毒化学品的复⽅制剂列⼊第⼆类精神药品管理。', 'answer': 'B', 'question_type': '最佳选择题', 'source': '2021年执业药师职业资格考试《药事管理与法规》'}


In [ ]:
your_prompt = """请回答下面的多选题，请直接正确答案选项，不要输出其他内容。
{question}
{options}"""

def get_query(da):
  da['options'] = '\n'.join([f"{k}:{v}" for k, v in da['option'].items() if v])
  return your_prompt.format_map(da)

for item in data:
  item['query'] = get_query(item)


print(data[0]['query'])

请回答下面的多选题，请直接正确答案选项，不要输出其他内容。
27. 根据国家药品监督管理局，公安部，国家卫⽣健康委员会的有关规定，⼜服固体制剂每剂量单位含羟考酮碱不超过5毫克，且不含其他⿇醉药品，精神药品或者药品类易制毒化学品的复⽅制剂列⼊（）。
A:含⿇醉药品复⽅制剂的管理
B:第⼆类精神药品管理
C:第⼀类精神药品管理
D:医疗⽤毒性药品管理


In [ ]:
model_answers = generate([item['query'] for item in data])
print(f'\n{model_answers[0]}')


B


In [ ]:
import re
from tqdm import tqdm

def get_ans(ans):
    match = re.findall(r'.*?([A-E]+(?:[、, ]+[A-E]+)*)', ans)
    if match:
        last_match = match[-1]
        return ''.join(re.split(r'[、, ，]+', last_match))
    return ''

correct_num = 0
total_num = 0
for model_answer, item in tqdm(zip(model_answers, data)):
  if get_ans(model_answer) == item['answer']:
    correct_num += 1
  total_num += 1
  item['model_answer'] = model_answer

print(f"ACC: {correct_num/total_num:.2%}")

result_path = "/content/result.jso"
with open(result_path, "w", encoding="utf-8") as file:
    json.dump(data, file, ensure_ascii=False, indent=4)
    print(f"Results are save in {result_path}")

20it [00:00, 6116.37it/s]

ACC: 50.00%
Results are save in /content/result.json
